**Notes On Readings**

The original study by Aungle & Langer (2023) is structured in which participants got standardized cupping induced bruises and were told 3 different “false” amounts of time had elapsed during healing observations: in half time, normal time, or double time. Blind raters judged the bruises to have healed significantly more in the condition where participants believed more time had passed. The authors framed this as evidence for the concept of “mind-body unity,” where subjective time perception affects recovery, irrespective of actual elapsed time.

Gelman & Brown are skeptical because the model let each person have their own starting point for healing but forced everyone to have the same treatment effect. Each individuals healing ability varies to some degree. The model assumed that the healing ability given time manipulation was a fixed number change, applied identically to everyone. Another issue they raised is the literatures used to support their argument were unreplicated, methdologically shaky, or simply mischaracterized.


## Garden of Forking Paths

Suppose there are $N$ different reasonable ways to analyze a dataset. The hypothesis $H$ is actually true, but each analysis has probability $q$ (Type I error rate) of incorrectly rejecting $H$.

### Part (a)

Assuming the $N$ outcomes are independent, what is the probability that at least one analysis incorrectly rejects $H$?

By the complement rule, $P(\text{at least one rejects}) = 1 - P(\text{none reject})$.

Each analysis fails to reject with probability $1 - q$, and since the outcomes are independent:

$$P(\text{none reject}) = (1-q)^N$$

Therefore:

$$\boxed{P(\text{at least one rejects}) = 1 - (1-q)^N}$$

In [ ]:
%pip install numpy matplotlib scipy

import numpy as np
import matplotlib.pyplot as plt

q = 0.05
N_vals = np.arange(1, 51)
prob = 1 - (1 - q) ** N_vals

plt.figure(figsize=(7, 4))
plt.plot(N_vals, prob, color='steelblue')
plt.axhline(0.5, color='gray', linestyle='--', label='50%')
plt.xlabel('Number of analyses N')
plt.ylabel('P(at least one false rejection)')
plt.title(f'Family-wise Type I error rate (q = {q})')
plt.legend()
plt.tight_layout()
plt.show()

### Part (b)

With $q = 0.05$, find $N$ such that $P(\text{at least one rejects}) \geq 0.5$:

$$1 - (0.95)^N = 0.5 \implies (0.95)^N = 0.5 \implies N = \frac{\log 0.5}{\log 0.95} \approx 13.51$$

So $N = 14$ analyses are enough to give a 50% chance of at least one false rejection.

In [3]:
q = 0.05
N_analytical = np.log(0.5) / np.log(1 - q)
print(f"Exact N: {N_analytical:.4f}  →  ceiling N = {int(np.ceil(N_analytical))}")

N = int(np.ceil(N_analytical))
print(f"Analytical P(at least one rejects) for N={N}: {1 - (1-q)**N:.4f}")

# Simulation
np.random.seed(42)
n_trials = 100_000
outcomes = np.random.random((n_trials, N)) < q
sim_prob = outcomes.any(axis=1).mean()
print(f"Simulated  P(at least one rejects) for N={N}: {sim_prob:.4f}")

Exact N: 13.5134  →  ceiling N = 14
Analytical P(at least one rejects) for N=14: 0.5123
Simulated  P(at least one rejects) for N=14: 0.5091


### Part (c)

**How does correlation between analyses change things?**

In reality the $N$ analysis choices share the same dataset, so their outcomes are **positively correlated** — analyses that are similar in structure will tend to agree.

- **High positive correlation** ($\rho \to 1$): all analyses give nearly identical results. The chance of at least one false rejection collapses back toward $q$ — no inflation beyond a single test.
- **Independence** ($\rho = 0$): the $1-(1-q)^N$ formula from part (a) applies.
- **Negative correlation**: analyses tend to disagree, which can push the false-rejection probability *above* the independent case, but true negative correlation among forking-path choices is unusual.

Real researcher degrees of freedom are positively correlated, so the true family-wise error rate is **between $q$ and $1-(1-q)^N$** — still inflated, but less so than the independent worst case.

**Simulation using a latent factor model:**  
Each analysis score is $X_i = \sqrt{\rho}\, Z + \sqrt{1-\rho}\, \varepsilon_i$ where $Z$ is a shared factor and $\varepsilon_i$ is independent noise. This gives $\text{Corr}(X_i, X_j) = \rho$. We reject when $X_i$ falls in the bottom $q$ quantile of the standard normal.

In [ ]:
import numpy as np
import scipy as sp

np.random.seed(42)
q = 0.05
N = 14
n_trials = 100_000
threshold = sp.special.ndtri(q)

print(f"Independent analytical (rho=0): {1 - (1-q)**N:.4f}")
print(f"Perfect correlation analytical (rho=1): {q:.4f}\n")
print(f"{'rho':>6}  {'P(at least one rejects)':>24}")
print("-" * 32)
for rho in [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]:
    Z = np.random.randn(n_trials)
    eps = np.random.randn(n_trials, N)
    X = np.sqrt(rho) * Z[:, None] + np.sqrt(1 - rho) * eps
    prob = (X < threshold).any(axis=1).mean()
    print(f"{rho:>6.1f}  {prob:>24.4f}")